# 🚀 01. Huấn luyện Multi-Modal Teacher Model (Giai đoạn 1B - CMKD)

> **Dự án:** Nhận diện quả sầu riêng chín hay không bằng Chưng cất tri thức Đa phương thức (CMKD)  
> **Phụ trách:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Vai trò của Notebook:** Notebook tinh gọn chỉ dùng để:
1. Kết nối Google Drive & kiểm tra GPU.
2. `git clone` / `git pull` mã nguồn mới nhất từ GitHub.
3. Liên kết dữ liệu (`data/final`) và trọng số (`checkpoints/unimodal/{vision,audio,sensor}`) từ Google Drive, đồng thời tạo thư mục đích `checkpoints/teachermodal`.
4. Kích hoạt thực thi file `train_teacher.py` từ thư mục gốc của dự án.

In [1]:
# ==============================================================================
# 1. KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Đã kết nối Google Drive thành công!")
else:
    print("ℹ️ Đang chạy trên môi trường Local.")

!nvidia-smi

✅ Đã kết nối Google Drive thành công!
Mon Nov  9 09:49:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------

In [2]:
# ==============================================================================
# 2. ĐỒNG BỘ CODE MỚI NHẤT TỪ GITHUB (NHÁNH cocobaut)
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        !git clone -b $BRANCH $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Gỡ symlink tạm trong repo (không ảnh hưởng dữ liệu trên Drive) để tránh xung đột khi git pull
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        !git checkout -f $BRANCH
        !git reset --hard origin/$BRANCH
else:
    # Nếu chạy local từ trong subfolder, tự lùi về thư mục gốc của repo
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Đang làm việc tại thư mục gốc dự án: {os.getcwd()}")

/content
Cloning into '/content/durian-ripeness-classification'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (322/322), done.
remote: Total 434 (delta 173), reused 354 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (434/434), 23.24 MiB | 19.52 MiB/s, done.
Resolving deltas: 100% (173/173), done.
/content/durian-ripeness-classification
✅ Đang làm việc tại thư mục gốc dự án: /content/durian-ripeness-classification


In [3]:
# ==============================================================================
# 3. CÀI ĐẶT THƯ VIỆN PHỤ THUỘC SIÊU TỐC VỚI `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Đã hoàn tất cài đặt toàn bộ thư viện!")

Using Python 3.13.16 environment at: /usr
Resolved 167 packages in 648ms                                       
Prepared 13 packages in 325ms                                            
Uninstalled 3 packages in 20ms
Installed 13 packages in 115ms                              
 + async-lru==2.4.0
 + comm==0.2.3
 - ipykernel==6.17.1
 + ipykernel==7.4.0
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.2.4
 - jupyter-client==7.4.9
 + jupyter-client==8.10.0
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.6
 + jupyterlab-server==2.28.1
 + nest-asyncio2==1.7.4
 - notebook==6.5.7
 + notebook==7.6.5
✅ Đã hoàn tất cài đặt toàn bộ thư viện!


In [4]:
# ==============================================================================
# 4. THIẾT LẬP ĐƯỜNG DẪN CHECKPOINTS & DATASET TỪ GOOGLE DRIVE
# ==============================================================================
# Folder Checkpoints trên Drive: https://drive.google.com/drive/folders/1K6II5aFG31kYrl7rsJj53u2Np4ZzjzG_?usp=drive_link
# Bên trong gồm: unimodal/{vision, audio, sensor} và teachermodal/
from pathlib import Path

if IN_COLAB:
    # Điều chỉnh đường dẫn này khớp với vị trí bạn lưu trên Google Drive
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS = DRIVE_PROJECT_ROOT / "results_teacher"

    # Tạo sẵn folder teachermodal & results_teacher trên Google Drive để lưu kết quả thời gian thực
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    # Tạo symlink từ repo tới Google Drive (mất mạng hay sập máy ảo Colab cũng không mất file)
    !rm -rf checkpoints/unimodal checkpoints/teachermodal workspaces/member1_vision_lead/04_train_teacher_and_student/results
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_RESULTS)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/teachermodal").mkdir(parents=True, exist_ok=True)
print("✅ Kiểm tra trạng thái đường dẫn:")
print(" - Unimodal dir   :", Path("checkpoints/unimodal").resolve())
print(" - Teachermodal   :", Path("checkpoints/teachermodal").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Kiểm tra trạng thái đường dẫn:
 - Unimodal dir   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/unimodal
 - Teachermodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Manifest CSV   : True


In [5]:
# ==============================================================================
# 5. THỰC THI HUẤN LUYỆN MULTI-MODAL TEACHER (CROSS-ATTENTION FUSION + COST-SENSITIVE FOCAL LOSS)
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_teacher.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --vision-ckpt checkpoints/unimodal/vision/best_vision_unimodal.pth \
    --audio-ckpt checkpoints/unimodal/audio/audio_backbone_teacher.pth \
    --sensor-ckpt checkpoints/unimodal/sensor/best_sensor_unimodal.pth \
    --save-path checkpoints/teachermodal/best_teacher.pth \
    --fusion-type cross_attention \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 8 \
    --epochs 30 \
    --warmup-epochs 5 \
    --eval-test \
    --device cuda


usage: train_teacher.py [-h] [--manifest MANIFEST] [--data-dir DATA_DIR]
                        [--sensor-cache-dir SENSOR_CACHE_DIR]
                        [--sensor-preprocessor SENSOR_PREPROCESSOR]
                        [--vision-ckpt VISION_CKPT] [--audio-ckpt AUDIO_CKPT]
                        [--sensor-ckpt SENSOR_CKPT] [--save-path SAVE_PATH]
                        [--results-dir RESULTS_DIR] [--epochs EPOCHS]
                        [--warmup-epochs WARMUP_EPOCHS]
                        [--batch-size BATCH_SIZE] [--accum-steps ACCUM_STEPS]
                        [--lr-head-warmup LR_HEAD_WARMUP]
                        [--lr-backbone LR_BACKBONE] [--lr-head LR_HEAD]
                        [--min-lr MIN_LR]
                        [--weight-decay-head WEIGHT_DECAY_HEAD]
                        [--weight-decay-backbone WEIGHT_DECAY_BACKBONE]
                        [--patience PATIENCE] [--grad-clip GRAD_CLIP]
                        [--label-smoothing LABEL_SMOOTHING]
 

In [ ]:
# ==============================================================================
# 6. ĐÁNH GIÁ ĐỘC LẬP & XUẤT BÁO CÁO BENCHMARK VỚI EVAL_TEACHER
# ==============================================================================
!python -m src.evaluators.eval_teacher \
    --checkpoint checkpoints/teachermodal/best_teacher.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --test-dir data/final/test \
    --input-mode multimodal \
    --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results

print("✅ Toàn bộ kết quả đánh giá đã được lưu tại:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results").resolve())